In [ ]:
#　日にちを指定して、ガーミンアプリからデータ（中程度運動　分、運動消費エネルギーkcal、歩数）を取得し、Outlook 予定表に貼り付ける
# 20251125確認済

In [1]:
# 期間を指定してデータを取得　2025/05/31確認 20250120以後のデータのみ

from datetime import datetime

def input_date():#日付入力の関数
    try:
        year = int(input("年を入力してください（例: 2025）："))
        month = int(input("月を入力してください（例: 5）："))
        day = int(input("日を入力してください（例: 31）："))
        dt = datetime(year, month, day)
        return dt.date()
    except ValueError as e:
        print("無効な日付です：", e)
        return None

def main():# 日付入力の関数
    dt_date = input_date() #日付入力関数を実行。2025/01/20以前のデータはない!!!
    return dt_date #開始日を返す

if __name__ == "__main__": # main()を実行する
    START_DATE = main()  # main() の戻り値を start_date に代入
    print("START_DATE:", START_DATE)
    if START_DATE:
        print("mainの外で表示：", START_DATE.strftime("%Y年%m月%d日"))

if __name__ == "__main__":
    END_DATE = main()  # main() の戻り値を end_date に代入
    print("END_DATE:", END_DATE)
    if END_DATE:
        print("mainの外で表示：", END_DATE.strftime("%Y年%m月%d日"))


START_DATE: 2026-03-01
mainの外で表示： 2026年03月01日
END_DATE: 2026-03-04
mainの外で表示： 2026年03月04日


In [2]:
# 20251124 12:57  Outlook連携（運動/消費/歩数/心拍 + 前日差つき）
from garminconnect import Garmin
from datetime import datetime, timedelta, time
import os
import re
import win32com.client

# =========================
# 設定
# =========================
SUBJECT_PREFIX = "[Garmin自動]"   # 今回はCategories判定なので必須ではないが残してOK
DISPLAY_FIX_OFFSET_HOURS = 9     # Outlook表示が-9hズラす前提の補正

email = os.environ["GARMIN_EMAIL"]
password = os.environ["GARMIN_PASSWORD"]

# =========================
# 1) Garminログイン
# =========================
garmin = Garmin(email, password)
garmin.login()

# =========================
# 2) Outlookカレンダー取得（アカウント指定）
# =========================
outlook = win32com.client.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")
calendar = namespace.Folders["kishi_isao@outlook.jp"].Folders["予定表"]

items = calendar.Items
items.IncludeRecurrences = True
items.Sort("[Start]")

# =========================
# Outlook終日イベント upsert
# =========================
def upsert_all_day_event(day_date, tag, subject_text):
    """
    day_date: datetime.date（その日）
    tag: "運動" / "消費" / "歩数" / "心拍" / "心拍_黄" / "心拍_赤"
    """

    # 0:00ではなく 9:00 をStartにする（表示補正用）
    start_local = datetime.combine(day_date, time(0, 0, 0)) + timedelta(hours=DISPLAY_FIX_OFFSET_HOURS)
    end_local   = start_local + timedelta(days=1)

    # 同日のアイテムを拾う（Restrictは“実日付”で探す）
    day_start_for_search = datetime.combine(day_date, time(0, 0, 0))
    day_end_for_search   = day_start_for_search + timedelta(days=1)

    restriction = (
        "[Start] >= '" + day_start_for_search.strftime("%m/%d/%Y 00:00 AM") + "' AND "
        "[Start] < '"  + day_end_for_search.strftime("%m/%d/%Y 00:00 AM") + "'"
    )
    day_items = items.Restrict(restriction)

    # 既存イベントを探す（Categoriesで判定）
    target = None
    for it in day_items:
        cats = str(it.Categories or "")

        # 心拍は "心拍" 系（心拍/心拍_黄/心拍_赤）を探す
        if tag.startswith("心拍"):
            if cats.startswith("心拍"):
                target = it
                break
        else:
            # 運動/消費/歩数 は Garmin自動;タグ のもの
            if ("Garmin自動" in cats) and (tag in cats):
                target = it
                break

    if target is None:
        target = calendar.Items.Add()

    # 絶対値で1日終日に固定
    target.AllDayEvent = True
    target.Start = start_local
    target.End   = end_local
    target.Duration = 1440
    target.BusyStatus = 0
    target.ReminderSet = False

    # Categories 設定
    if tag.startswith("心拍"):
        # 例: "心拍" / "心拍_黄" / "心拍_赤"
        target.Categories = tag
    else:
        target.Categories = f"Garmin自動;{tag}"

    target.Subject = subject_text
    target.Body = ""
    target.Save()


# =========================
# 前日のRHRをOutlookから取得
# =========================
def get_prev_rhr_from_outlook(prev_date):
    """
    前日prev_dateの心拍イベント（Categoriesが心拍で始まる）から Subject の
    'RHR 46...' の 46 を抜き出す。無ければNone。
    """
    day_start = datetime.combine(prev_date, time(0, 0, 0))
    day_end   = day_start + timedelta(days=1)

    restriction = (
        "[Start] >= '" + day_start.strftime("%m/%d/%Y 00:00 AM") + "' AND "
        "[Start] < '"  + day_end.strftime("%m/%d/%Y 00:00 AM") + "'"
    )
    day_items = items.Restrict(restriction)

    found = None
    for it in day_items:
        cats = str(it.Categories or "")
        if cats.startswith("心拍"):
            subj = str(it.Subject or "")
            m = re.search(r"RHR\s*(\d+)", subj)
            if m:
                found = int(m.group(1))
                # 複数あっても最後に見つかったものを採用（重複耐性）
    return found


# =========================
# 3) 指定期間のGarminデータをOutlook終日イベントに反映
# =========================
# START_DATE, END_DATE は前のセルで date 型として入力している前提
start_date = datetime.combine(START_DATE, datetime.min.time())
end_date   = datetime.combine(END_DATE,   datetime.min.time())

current = start_date
while current <= end_date:
    date_str = current.strftime("%Y-%m-%d")
    day_date = current.date()

    # =========================================================
    # 1) Garmin 日次データ（歩数/消費/運動分）
    # =========================================================
    stats = garmin.get_stats(date_str) or {}

    steps = stats.get("totalSteps", 0) or 0
    calories = stats.get("activeKilocalories", 0) or 0
    moderate_minutes = stats.get("moderateIntensityMinutes") or 0
    vigorous_minutes = stats.get("vigorousIntensityMinutes") or 0
    exercise_minutes = moderate_minutes + vigorous_minutes

    exercise_minutes_i = int(exercise_minutes or 0)
    calories_i = int(calories or 0)
    steps_i = int(steps or 0)

    ex_str = f"{exercise_minutes_i:03d}"
    cal_str = f"{calories_i:04d}"
    steps_str = f"{steps_i:05d}"

    # =========================================================
    # 2) Garmin 心拍データ（RHR/MAX/MIN/RHR7）
    # =========================================================
    try:
        hr = garmin.get_heart_rates(date_str) or {}
    except Exception as e:
        print(f"⚠️ {date_str} 心拍取得に失敗: {e}")
        hr = {}

    rhr = int(hr.get("restingHeartRate") or 0)
    hr_max = int(hr.get("maxHeartRate") or 0)
    hr_min = int(hr.get("minHeartRate") or 0)
    rhr7 = int(hr.get("lastSevenDaysAvgRestingHeartRate") or 0)

    # =========================================================
    # 3) アラート判定（黄/赤/通常）
    # =========================================================
    delta = rhr - rhr7  # 今日RHRが7日平均よりどれだけ高いか

    Y_RHR_DELTA = 8
    R_RHR_DELTA = 12
    Y_MAX = 120
    R_MAX = 135

    level = 0
    if rhr7 > 0:
        if delta >= R_RHR_DELTA:
            level = 2
        elif delta >= Y_RHR_DELTA:
            level = 1

    if hr_max >= R_MAX:
        level = 2
    elif hr_max >= Y_MAX and level < 2:
        level = 1

    if level == 2:
        category_tag = "心拍_赤"
    elif level == 1:
        category_tag = "心拍_黄"
    else:
        category_tag = "心拍"

    # =========================================================
    # 4) 前日差（Outlookの前日イベントから）
    # =========================================================
    prev_rhr = get_prev_rhr_from_outlook(day_date - timedelta(days=1))

    if (prev_rhr is None) or (rhr == 0):
        diff_txt = ""
    else:
        diff_prev = rhr - prev_rhr
        if diff_prev > 0:
            diff_txt = f"(+{diff_prev})"
        elif diff_prev < 0:
            diff_txt = f"({diff_prev})"
        else:
            diff_txt = "(0)"

    hr_all = f"RHR {rhr}{diff_txt} | MAX {hr_max} | MIN {hr_min} | 7d {rhr7}"

    # =========================================================
    # 5) Outlook 終日イベント反映（upsert）
    # =========================================================
    upsert_all_day_event(day_date, "運動", f"{ex_str}分")
    upsert_all_day_event(day_date, "消費", f"運動消費{cal_str}kcal")
    upsert_all_day_event(day_date, "歩数", f"歩数{steps_str}")
    upsert_all_day_event(day_date, category_tag, hr_all)

    print(
        f"✅ {date_str} 反映："
        f"運動{exercise_minutes_i}分 / {calories_i}kcal / {steps_i}歩 / "
        f"{hr_all}"
    )

    current += timedelta(days=1)

print("=== 完了しました ===")


Rebuilding cache of generated files for COM support...
Checking 00062FFF-0000-0000-C000-000000000046x0x9x6
Could not add module (IID('{00062FFF-0000-0000-C000-000000000046}'), 0, 9, 6) - <class 'AttributeError'>: module 'win32com.gen_py.00062FFF-0000-0000-C000-000000000046x0x9x6' has no attribute 'CLSIDToClassMap'
Done.
✅ 2026-03-01 反映：運動73分 / 152kcal / 7277歩 / RHR 52(-5) | MAX 103 | MIN 49 | 7d 60
✅ 2026-03-02 反映：運動29分 / 154kcal / 6660歩 / RHR 50(-2) | MAX 111 | MIN 47 | 7d 59
✅ 2026-03-03 反映：運動73分 / 226kcal / 7740歩 / RHR 46(-4) | MAX 111 | MIN 45 | 7d 57
✅ 2026-03-04 反映：運動16分 / 38kcal / 187歩 / RHR 59(+13) | MAX 108 | MIN 63 | 7d 56
=== 完了しました ===


In [ ]:
# =========================
# ★ 心拍データ確認（1日だけテスト）
# =========================

test_date = START_DATE.strftime("%Y-%m-%d")  # 既に指定している開始日でOK

print(f"\n--- 心拍データ確認：{test_date} ---")

try:
    hr_data = garmin.get_heart_rates(test_date)
    print("型:", type(hr_data))
    if hasattr(hr_data, "keys"):
        print("キー一覧:", hr_data.keys())
    print("中身サンプル:", hr_data)
except Exception as e:
    print("取得エラー:", e)

In [ ]:
g.get_sleep_data("2026-02-18")


In [ ]:
print(data.keys())


In [ ]:
from garminconnect import Garmin
import os

email = os.environ["GARMIN_EMAIL"]
password = os.environ["GARMIN_PASSWORD"]

g = Garmin(email, password)
g.login()

print("ログイン成功")